In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================
# 1. VERİ YÜKLEME
# =========================
train = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/train.csv')
test = pd.read_csv('/kaggle/input/competitions/yzta-2026-datathon/test_x.csv')
test_ids = test['id']

# =========================
# 2. İSİM DÜZELTMELERİ
# =========================
ulke_map = {'Spain': 'Ispanya', 'South Mexico': 'Meksika',
            'Netherlands': 'Hollanda', 'Korea': 'Guney Kore'}
train['ulke'] = train['ulke'].replace(ulke_map)
test['ulke'] = test['ulke'].replace(ulke_map)
train['meslek'] = train['meslek'].replace({'Lawyer': 'Avukat'})
test['meslek'] = test['meslek'].replace({'Lawyer': 'Avukat'})

# =========================
# 3. EKSİK DEĞER YÖNETİMİ
# =========================
train = train.dropna(subset=['kronotip', 'meslek', 'ruh_sagligi_durumu', 'bilissel_performans_skoru'])

# Meslek bazlı doldur
for col in ['stres_skoru', 'vucut_kitle_indeksi']:
    meslek_ort = train.groupby('meslek')[col].mean()
    train[col] = train[col].fillna(train['meslek'].map(meslek_ort))
    test[col] = test[col].fillna(test['meslek'].map(meslek_ort))

sayisal_sutunlar = ['vucut_kitle_indeksi', 'uyku_oncesi_kafein_mg',
                     'stres_skoru', 'derin_uyku_yuzdesi',
                     'uykuya_dalma_suresi_dk', 'gecelik_uyanma_sayisi',
                     'uyku_oncesi_ekran_suresi_dk', 'gunluk_adim_sayisi',
                     'sekerleme_suresi_dk', 'gunluk_calisma_saati',
                     'dinlenik_nabiz_bpm', 'oda_sicakligi_celsius',
                     'hafta_sonu_uyku_farki_saat']

train[sayisal_sutunlar] = train[sayisal_sutunlar].fillna(train[sayisal_sutunlar].median())
test['uyku_oncesi_kafein_mg'] = test['uyku_oncesi_kafein_mg'].fillna(train['uyku_oncesi_kafein_mg'].median())
test['vucut_kitle_indeksi'] = test['vucut_kitle_indeksi'].fillna(train['vucut_kitle_indeksi'].median())
test['stres_skoru'] = test['stres_skoru'].fillna(train['stres_skoru'].median())
for col in ['kronotip', 'meslek', 'ruh_sagligi_durumu']:
    test[col] = test[col].fillna(train[col].mode()[0])

# =========================
# 4. TARGET ENCODING
# =========================
kategorik_sutunlar = ['cinsiyet', 'meslek', 'ulke', 'kronotip',
                       'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']

for col in kategorik_sutunlar:
    mean_map = train.groupby(col)['bilissel_performans_skoru'].mean()
    train[col + '_target'] = train[col].map(mean_map)
    test[col + '_target'] = test[col].map(mean_map)

# =========================
# 5. LABEL ENCODING
# =========================
le = LabelEncoder()
for col in kategorik_sutunlar:
    combined = pd.concat([train[col], test[col]], axis=0).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# =========================
# 6. FEATURE ENGINEERING
# =========================
for veri in [train, test]:
    veri['stres_calisma'] = veri['stres_skoru'] * veri['gunluk_calisma_saati']
    veri['toplam_uyku_kalitesi'] = veri['rem_yuzdesi'] + veri['derin_uyku_yuzdesi']
    veri['stres_ekran'] = veri['stres_skoru'] * veri['uyku_oncesi_ekran_suresi_dk']

test['stres_calisma'] = test['stres_calisma'].fillna(test['stres_skoru'] * test['gunluk_calisma_saati'])
test['stres_ekran'] = test['stres_ekran'].fillna(test['stres_skoru'] * test['uyku_oncesi_ekran_suresi_dk'])

print("Train:", train.shape)
print("Test:", test.shape)
print("Eksik train:", train.isnull().sum().sum())
print("Eksik test:", test.isnull().sum().sum())

# =========================
# 7. MODEL
# =========================
X = train.drop(columns=['id', 'bilissel_performans_skoru'])
y = train['bilissel_performans_skoru']
X_test = test.drop(columns=['id'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Optuna CatBoost
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_seed': 42, 'verbose': 0
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(objective_cat, n_trials=30)
print(f"CatBoost RMSE: {study_cat.best_value:.4f}")

# Optuna XGBoost
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42, 'verbosity': 0
    }
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30)
print(f"XGBoost RMSE: {study_xgb.best_value:.4f}")

# =========================
# 8. TÜMÜ İLE EĞİT
# =========================
cat_final = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=0)
cat_final.fit(X, y)

xgb_final = XGBRegressor(**study_xgb.best_params, random_state=42, verbosity=0)
xgb_final.fit(X, y)

lgbm_final = LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
lgbm_final.fit(X, y)

pred_final = (cat_final.predict(X_test) * 0.70) + (xgb_final.predict(X_test) * 0.25) + (lgbm_final.predict(X_test) * 0.05)
pred_final = np.clip(pred_final, 0, 10)

submission = pd.DataFrame({
    'id': test_ids,
    'bilissel_performans_skoru': pred_final
})

submission.to_csv('submission.csv', index=False)
print("Hazır!")
print(submission.shape)